# Kestrel M1 · 02 Metrics

The four Milestone 1 metrics (numbers 1, 2, 8 and 9 of the overview), computed **in pandas over the curated
output** of the pipeline (`output/curated/*.parquet`, i.e. accepted rows after dedup and rejects).

Every metric is reported with the policy behind it (Dataset Description, Appendix A): **denominator, population
filter, late-event policy and null policy**. The computations live in `kestrel_m1/metrics.py`; this notebook only
calls them, shows the numbers, and explains the choices. Values are written to `output/metrics/`.

| # | Metric | Appendix A definition |
|---|---|---|
| 1 | session volume · success · error rate | distinct `session_id` with a terminal event; successful, resp. error, terminal sessions / all terminal |
| 2 | p50 / p95 session duration | percentile of derived end-to-end duration for completed (terminal) sessions, linear interpolation |
| 8 | trace completeness · correlation coverage | terminal sessions with ≥1 `llm.turn` and no gap in accepted `sequence_number`s; SIEM alerts matched to a session |
| 9 | rejected · late-event rate | rejected / received; late-accepted / accepted (late = `ingest_time − event_time` > 10 s) |

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> M1/
sys.path.insert(0, str(ROOT))

import pandas as pd
from kestrel_m1 import metrics, pipeline
from kestrel_m1.rejects import REASONS
from kestrel_m1.timeutil import parse_activity_time, parse_iso_utc, lateness_seconds, LATE_THRESHOLD_S

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# a clean clone has no curated output yet: build it (Parquet only, ~40 s)
if not (metrics.CURATED / "canonical_activity.parquet").exists():
    pipeline.main(["--no-db"])
t = metrics.load_curated()
act = t["stg_agent_activity"]
{name: len(df) for name, df in t.items()}

{'stg_agent_activity': 924775,
 'stg_network_flows': 42308,
 'stg_siem_alerts': 3754,
 'siem_correlation': 3754,
 'correlation_sensitivity': 11,
 'stage_summary': 15,
 'fact_rejected_record': 13589}

## 0. Can the curated output be trusted? (invariants)

Before computing anything, check the properties the metrics rely on:

* every received record is either accepted or rejected, per source (nothing silently dropped)
* every reject has exactly one reason from the closed enum, plus its source file and line
* the canonical table has a unique `activity_id` and a UTC `event_time_utc` on every row

In [2]:
s = t["stage_summary"]
per_source = s.assign(kind=s.outcome.where(s.outcome == "ACCEPTED", "REJECTED")).pivot_table(
    index="source_file", columns="kind", values="rows", aggfunc="sum", fill_value=0)
per_source["received"] = per_source.sum(axis=1)
canon = pd.read_parquet(metrics.CURATED / "canonical_activity.parquet",
                        columns=["activity_id", "source_file", "source_line", "event_time_utc"])
rej = t["fact_rejected_record"]

assert (per_source.ACCEPTED == canon.source_file.value_counts().reindex(per_source.index)).all()
assert (per_source.REJECTED == rej.source_file.value_counts().reindex(per_source.index, fill_value=0)).all()
assert rej.reason.isin(REASONS).all() and rej[["source_file", "source_line"]].notna().all().all()
assert not rej.duplicated(["source_file", "source_line"]).any()       # one reason per record
assert canon.activity_id.is_unique and canon.event_time_utc.notna().all()
assert str(canon.event_time_utc.dtype).endswith("UTC]")
print("all invariants hold")
per_source

all invariants hold


kind,ACCEPTED,REJECTED,received
source_file,,,
agent_activity.parquet,924775,12977,937752
network_flows.csv,42308,84,42392
sandbox_audit.log,183477,528,184005
siem_alerts.json,3754,0,3754


## Shared population: terminal sessions

Metrics 1, 2 and 8a share one population, built once by `metrics.session_outcomes`: one row per session with an
accepted `session.end` or `session.error` row (**terminal session**, Appendix A conventions). Checked on the data:

* no session has more than one terminal row, and none has both an end and an error (asserted in the function)
* every terminal session has its accepted `session.start` (the orphan rule rejects rows whose session has none)
* sessions without any terminal row are **outside** the population: they never finished in the export window, so
  they have no outcome and no duration

In [3]:
o = metrics.session_outcomes(act)
sessions = act.session_id.nunique()
pd.Series({"sessions with an accepted start": sessions,
           "terminal sessions (population)": len(o),
           "sessions without a terminal row (excluded)": sessions - len(o),
           "terminal sessions without an accepted start": int(o.start_time_utc.isna().sum())})

sessions with an accepted start                100000
terminal sessions (population)                  97952
sessions without a terminal row (excluded)       2048
terminal sessions without an accepted start         0
dtype: int64

In [4]:
o.head()

,session_id,outcome,start_time_utc,end_time_utc,terminal_latency_s,duration_s,latency_disagrees,n_events,n_seq,min_seq,max_seq,n_turns,has_gap,trace_complete
0,ss_017007,success,2026-01-31 00:00:33.753000+00:00,2026-01-31 00:00:42.222000+00:00,8.469,8.469,False,5,5,1,5,2,False,True
1,ss_060876,success,2026-01-31 00:00:48.906000+00:00,2026-01-31 00:01:06.949000+00:00,18.043,18.043,False,13,13,1,13,6,False,True
2,ss_000310,success,2026-01-31 00:01:57.997000+00:00,2026-01-31 00:02:17.524000+00:00,19.527,19.527,False,9,9,1,9,4,False,True
3,ss_083538,success,2026-01-31 00:02:11.250000+00:00,2026-01-31 00:02:26.572000+00:00,15.322,15.322,False,7,7,1,7,3,False,True
4,ss_023376,success,2026-01-31 00:03:06.237000+00:00,2026-01-31 00:03:33.584000+00:00,27.347,27.347,False,13,13,1,13,6,False,True


## Metric 1 · Session volume, success rate, error rate

* **Denominator:** distinct `session_id` with an accepted terminal row.
* **Population:** accepted activity rows only (after dedup on `event_id` and the reject ladder).
* **Late policy:** late rows are accepted and only flagged `LATE` (the batch applies no watermark), so a session whose
  terminal row arrived late is still terminal.
* **Null policy:** none possible: `event_type`, `session_id` and `event_time` are required; rows missing them are rejected.
* **Success / error:** `session.end` = success, `session.error` = error. The terminal row's `status` field is *not*
  used: the definition is the terminal event type.

In [5]:
m1 = metrics.metric_1(o)
m1

,metric,value,numerator,denominator,unit
0,1 terminal sessions (volume),"97,952.000",<NA>,<NA>,sessions
1,1 success rate,92.506,90611,97952,%
2,1 error rate,7.494,7341,97952,%


## Metric 2 · p50 / p95 session duration

* **Formula:** duration = terminal `event_time_utc` − `session.start` `event_time_utc` (Dataset Description 3.5),
  in seconds. `latency_ms` is **not** the duration; it is used only as a cross-check below.
* **Denominator / population:** the terminal sessions of metric 1 (all of them have a start, so none is dropped).
* **Percentiles:** `Series.quantile(q, interpolation="linear")`: linear interpolation between closest ranks, as Appendix A requires.
* **Late policy:** included. Lateness is about `ingest_time`; the duration uses `event_time` only, so a late
  arrival does not change a duration.
* **Null policy:** a session without an accepted start would have no duration and be excluded (none in this data).

In [6]:
m2 = metrics.metric_2(o)
m2

,metric,value,numerator,denominator,unit
0,2 p50 session duration,17.646,<NA>,97952,s
1,2 p95 session duration,39.560,<NA>,97952,s
2,2 cross-check: terminal latency_ms disagrees > 1 s,1.983,1942,97952,%


In [7]:
o.duration_s.describe(percentiles=[.05, .25, .5, .75, .95, .99])

count   97,952.000
mean        19.692
std         10.188
min          2.335
5%           7.529
25%         12.529
50%         17.646
75%         24.379
95%         39.560
99%         54.367
max        109.492
Name: duration_s, dtype: float64

### Cross-check: terminal `latency_ms` vs derived duration

The spec asks how often the two disagree. On most terminal rows `latency_ms` equals the derived duration to the
millisecond; on the rest it differs by up to ±30 s in both directions, so it cannot replace the derived duration.

In [8]:
diff = (o.terminal_latency_s - o.duration_s)
pd.DataFrame({"disagreeing sessions (> 1 s)": [int(o.latency_disagrees.sum())],
              "share %": [round(100 * o.latency_disagrees.mean(), 3)],
              "exactly equal (< 1 ms)": [int((diff.abs() < 0.001).sum())],
              "min diff s": [diff.min()], "max diff s": [diff.max()]})

,disagreeing sessions (> 1 s),share %,exactly equal (< 1 ms),min diff s,max diff s
0,1942,1.983,96010,-29.979,29.989


## Metric 8a · Trace completeness

* **Definition (Appendix A):** a terminal session with at least one `llm.turn` and **no gap in its accepted
  `sequence_number`s**. Implemented as: ≥1 accepted `llm.turn`, and the accepted sequence numbers are exactly
  `1..max` (no hole, no repeat).
* **Denominator:** terminal sessions (metric 1 population).
* **Late policy:** late rows are accepted, so they fill their sequence slot.
* **Null policy:** `sequence_number` is required; a row without it is rejected and therefore leaves a gap.
* **Consequence of "accepted":** a row rejected for any reason (except `DUPLICATE_EVENT_ID`, which leaves its kept
  copy) leaves a hole, so the session becomes incomplete. That is intentional: the curated trace really is missing
  that step.

In [9]:
m8a = metrics.metric_8_trace(o)
m8a

,metric,value,numerator,denominator,unit
0,8 trace completeness,95.881,93917,97952,%


Why sessions are incomplete. For every gap the missing sequence numbers are compared with the rejected rows'
`(session_id, sequence_number)`: most gaps were left by our own rejects, a smaller part was never sent by the platform.

In [10]:
metrics.trace_breakdown(act, o, rej)

,trace_status,sessions,share_pct
0,complete,93917,95.881
1,gap left by rejected rows,3540,3.614
2,gap in source (never sent),472,0.482
3,gap: both causes,22,0.022
4,no llm.turn,1,0.001


## Metric 8b · Correlation coverage

* **Definition:** SIEM alerts matched to a session (correlation rule and N = 60 s from `kestrel_m1/correlate.py`:
  `src_ip = source_ip`, `dst_ip = destination.host`, `dst_port = destination.port`, `|Δt| ≤ 60 s`).
* **Denominator:** all accepted SIEM alerts.
* **Numerator (declared):** alerts linked to **exactly one session**: every `matched` alert (one candidate flow), plus
  `ambiguous` alerts whose candidate flows all belong to the same session (the flow is uncertain, the session is
  not). An ambiguous alert spanning several sessions, and every unmatched alert, is not covered. Nothing is forced.
* **Late policy:** not applicable (alerts and flows carry no `ingest_time`); the time rule is `|Δt| ≤ N`.
* **Null policy:** an alert without a session is uncovered.

Two variants are shown so the choice is transparent: *strict* (single-flow matches only), and *on-platform* (only
alerts whose `source_ip` belongs to some Kestrel sandbox; cloud-ids `*_external` alerts come from IPs no sandbox
ever used, so they cannot be matched by construction).

In [11]:
m8b = metrics.metric_8_coverage(t["siem_correlation"], t["stg_siem_alerts"], t["stg_network_flows"])
m8b

,metric,value,numerator,denominator,unit
0,8 correlation coverage (alert -> session),64.864,2435,3754,%
1,"8 correlation coverage, strict (single-flow match only)",64.172,2409,3754,%
2,"8 correlation coverage, on-platform alerts only",97.283,2435,2503,%


In [12]:
c = t["siem_correlation"].merge(t["stg_siem_alerts"], on="alert_id")
c["covered"] = c.session_id.notna()
pd.crosstab([c.source_system], [c.match_status, c.covered], margins=True)

match_status  ambiguous      matched unmatched   All
covered           False True    True     False      
source_system                                       
cloud-ids             0    0       0      1251  1251
corp-siem             0   24    1843        45  1912
egress-proxy          1    2     566        22   591
All                   1   26    2409      1318  3754

Coverage at the other tolerances (`correlation_sensitivity`): the counts are flat from 45 s to 120 s, which is why
N = 60 s was declared. Larger N only adds alerts whose nearest flow is minutes away.

In [13]:
sens = t["correlation_sensitivity"].copy()
sens

,n_s,matched,ambiguous,unmatched,alerts_with_session
0,5,285,2,3467,287
1,10,579,4,3171,583
2,15,862,6,2886,868
3,30,1654,16,2084,1670
4,45,2409,27,1318,2435
5,60,2409,27,1318,2435
6,90,2409,27,1318,2435
7,120,2409,27,1318,2435
8,300,2453,27,1274,2479
9,600,2475,28,1251,2501


## Metric 9a · Rejected-record rate

* **Definition:** rejected / received, per source. Received = accepted + rejected (asserted in section 0).
* **Headline:** `agent_activity.parquet`, the main input. The pooled rate over the four event sources is also given.
  Reference files are cleaned by dedup and repair (documented in the DQ findings), not rejected, so they are not counted.
* **Late policy:** the batch never rejects for lateness (`LATE_BEYOND_WATERMARK` is reserved for the M2 stream with its
  60 s watermark), so lateness does not raise this rate.
* **Null policy:** a null in a required field is a reject (`MISSING_REQUIRED_FIELD`) and is counted.

In [14]:
m9a = metrics.metric_9_rejected(t["stage_summary"])
m9a

,metric,value,numerator,denominator,unit
0,9 rejected rate: agent_activity.parquet,1.384,12977,937752,%
1,9 rejected rate: network_flows.csv,0.198,84,42392,%
2,9 rejected rate: sandbox_audit.log,0.287,528,184005,%
3,9 rejected rate: siem_alerts.json,0.000,0,3754,%
4,9 rejected rate: all four sources,1.164,13589,1167903,%


In [15]:
metrics.rejects_by_reason(rej).pivot_table(index="reason", columns="source_file", values="rejected",
                                           observed=True, fill_value=0).astype("int64")

source_file,agent_activity.parquet,network_flows.csv,sandbox_audit.log
reason,,,
DUPLICATE_EVENT_ID,7427,0,4
UNKNOWN_AGENT_ID,1332,0,0
UNKNOWN_TENANT_ID,730,0,0
MALFORMED_TIMESTAMP,708,0,0
ORPHAN_EVENT,200,84,0
INVALID_EVENT_TYPE,302,0,0
MISSING_REQUIRED_FIELD,730,0,0
INVALID_NUMERIC_RANGE,1548,0,0
UNPARSEABLE_RECORD,0,0,524


## Metric 9b · Late-event rate

* **Definition:** late-accepted / accepted, late = `ingest_time_utc − event_time_utc` > 10 s.
* **Population:** accepted `agent_activity` rows. It is the only source with an `ingest_time`; flows, the audit log
  and SIEM alerts have no arrival time, so no lateness can be measured for them.
* **Late policy:** this metric *is* the late policy's measurement: late rows are kept and flagged `LATE`.
* **Null policy:** none possible: both timestamps are required and a malformed one is rejected.

In [16]:
m9b = metrics.metric_9_late(act)
m9b

,metric,value,numerator,denominator,unit
0,9 late-event rate (lateness > 10 s),1.499,13865,924775,%


In [17]:
metrics.late_by_event_type(act)

,event_type,accepted,late,p99_lateness_s,max_lateness_s,late_pct
0,llm.turn,407462,6080,965.793,"2,700.902",1.492
1,session.end,90611,1382,"1,021.894","2,701.158",1.525
2,session.error,7341,93,573.308,"2,663.454",1.267
3,session.start,100000,1486,990.435,"2,700.566",1.486
4,tool.call,319361,4824,998.924,"2,700.641",1.511


### Why the rate on the curated table is lower than on the raw file

Profiling counted lateness on the **raw** rows. Duplicate deliveries are re-sends, so they arrive later than the
original; dedup keeps the copy with the earliest `ingest_time` and rejects the others. Those late copies are therefore
not in the accepted population. This cell measures it from the rejected duplicates' payloads:

In [18]:
dup = rej[(rej.source_file == "agent_activity.parquet") & (rej.reason == "DUPLICATE_EVENT_ID")]
p = pd.DataFrame(dup.raw_payload.map(json.loads).tolist())
dup_late = lateness_seconds(parse_iso_utc(p.ingest_time), parse_activity_time(p.event_time).ts) > LATE_THRESHOLD_S
late_acc, n_acc = int(act.is_late.sum()), len(act)
pd.DataFrame({
    "population": ["accepted rows (declared metric)", "rejected duplicate copies", "accepted + duplicate copies"],
    "rows": [n_acc, len(dup), n_acc + len(dup)],
    "late": [late_acc, int(dup_late.sum()), late_acc + int(dup_late.sum())],
}).assign(late_pct=lambda d: (100 * d.late / d.rows).round(3))

,population,rows,late,late_pct
0,accepted rows (declared metric),924775,13865,1.499
1,rejected duplicate copies,7427,7329,98.680
2,accepted + duplicate copies,932202,21194,2.274


## Late-event policy: what it changes

Declared policy: late rows are **kept** (flagged `LATE`). As a sensitivity check only, metrics 1, 2 and 8a are
recomputed with every late row dropped. Durations and rates barely move, but trace completeness falls sharply,
because each dropped row leaves a hole in its session's sequence. That is the main reason to keep late rows in batch.

In [19]:
metrics.late_policy_effect(act)

,metric,late kept (declared),late dropped,difference
0,1 terminal sessions (volume),"97,952.000","96,477.000","-1,475.000"
1,1 success rate,92.506,92.487,-0.019
2,1 error rate,7.494,7.513,0.019
3,2 p50 session duration,17.646,17.635,-0.011
4,2 p95 session duration,39.560,39.613,0.053
5,8 trace completeness,95.881,84.696,-11.185


## Missing token counts: declared choice and its effect

Some accepted `llm.turn` rows have no `output_tokens` (flag `OUTPUT_TOKENS_MISSING`). **Declared policy:
null-excluded.** The turn is kept, its `tokens_out` and `cost_usd` stay null and are left out of sums and averages;
they are never imputed as 0, because a turn that produced text did not produce zero tokens.

Effect on two token metrics, null-excluded vs zero-imputed. Zero-imputation looks like it gives *more* turns a cost
and a *higher* total, but it adds input-only costs for turns whose output was clearly not free, and it drags the
per-turn averages down.

Separately from this policy, turns on `mdl_kst_nano_1` have no price at all (flag `PRICE_MISSING`): their cost is
null under either choice.

In [20]:
metrics.missing_token_effect(act)

,quantity,null_excluded (declared),zero_imputed,difference_pct
0,llm.turn rows,"407,462.000","407,462.000",0.000
1,turns with output_tokens,"403,364.000","407,462.000",1.016
2,mean output_tokens per turn,239.800,237.388,-1.006
3,total output_tokens,"96,726,648.000","96,726,648.000",0.000
4,turns with a cost,"374,978.000","378,799.000",1.019
5,"estimated LLM cost, USD","4,055.110","4,087.310",0.794
6,"mean cost per 1,000 costed turns, USD",10.814,10.790,-0.222


## Summary: the M1 metrics and their policies

Written to `output/metrics/m1_metrics.csv` (values), `m1_metric_policies.csv` (one row of policy per metric) and
`missing_token_effect.csv`. The same files are produced by `python -m kestrel_m1.metrics`.

In [21]:
values = metrics.write_outputs(t)
values

,metric,value,numerator,denominator,unit
0,1 terminal sessions (volume),"97,952.000",<NA>,<NA>,sessions
1,1 success rate,92.506,90611,97952,%
2,1 error rate,7.494,7341,97952,%
3,2 p50 session duration,17.646,<NA>,97952,s
4,2 p95 session duration,39.560,<NA>,97952,s
5,2 cross-check: terminal latency_ms disagrees > 1 s,1.983,1942,97952,%
6,8 trace completeness,95.881,93917,97952,%
7,8 correlation coverage (alert -> session),64.864,2435,3754,%
8,"8 correlation coverage, strict (single-flow match only)",64.172,2409,3754,%
9,"8 correlation coverage, on-platform alerts only",97.283,2435,2503,%


In [22]:
metrics.POLICIES

,metric,denominator,population,late_policy,null_policy
0,1 session volume / success / error rate,distinct session_id with an accepted session.end or session.error row,accepted activity rows (after dedup and rejects). Every terminal session has an accepted session.start because orpha...,"late rows are accepted and only flagged (the batch has no watermark), so a late terminal row still makes its session...","none possible: event_type, session_id and event_time are required fields, and rows missing them are rejected"
1,2 p50 / p95 session duration,terminal sessions (same population as metric 1),duration = terminal event_time_utc - session.start event_time_utc; pandas Series.quantile with linear interpolation ...,"included. Lateness concerns ingest_time, and the duration uses event_time only",a session without an accepted start would have no duration and be excluded; the curated data has none. latency_ms is...
2,8a trace completeness,terminal sessions (same population as metric 1),"complete = at least one accepted llm.turn, and the accepted sequence_numbers are exactly 1..max (no gap, no repeat)....","late rows are accepted, so they fill their sequence slot","sequence_number is required; rows without it are rejected, which leaves a gap"
3,8b correlation coverage,all accepted SIEM alerts,covered = the alert is linked to exactly one session at N = 60 s: matched (one candidate flow) plus ambiguous alerts...,not applicable: alerts and flows carry no ingest_time; |delta t| <= N is the only time rule,an alert without a session is not covered; it is never force-matched
4,9a rejected-record rate,received records per source = accepted + rejected (every received record lands in exactly one of the two; the pipeli...,the four event sources. Headline = agent_activity.parquet; the overall rate is pooled over all four. Reference files...,"late rows are not rejected in batch (no LATE_BEYOND_WATERMARK), so lateness does not raise this rate",a null in a required field is a reject (MISSING_REQUIRED_FIELD) and is counted
5,9b late-event rate,accepted agent_activity rows,"late = ingest_time_utc - event_time_utc > 10 s. Only agent_activity has an ingest_time, so the other sources have no...",this metric is the late policy's measurement: late rows are kept and flagged LATE,none possible: both timestamps are required and a malformed one is a reject
6,missing token counts,llm.turn rows,turns whose output_tokens is null,-,"null-excluded: the count stays null, cost_usd is null and both are left out of sums and averages, never imputed as 0..."
